# Data Splitting:

### Train / Val / Test split (70 / 10 / 20)

- split_real_fake / train, val, test / real, fake
- split_gender    / train, val, test / male, female, unknown
- split_age       / train, val, test / under_18, 18-25, 26-35, 36-50, 50+

# Import Libraries:

In [1]:
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

## Step 1: Paths and load the csv

In [2]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

UNIQUE_CSV = DATA_DIR / "unique_dataset.csv"
UNIQUE_DIR = DATA_DIR / "unique_raw_images"

if not UNIQUE_CSV.exists():
    raise FileNotFoundError("Mudhalla Notebook 1 run pannunga. Idhu illa: " + str(UNIQUE_CSV))

df = pd.read_csv(UNIQUE_CSV)
df["label"] = df["label"].str.lower()       # REAL -> real
df["gender"] = df["gender"].str.lower()     # Male -> male
print("Total unique images:", len(df))

Total unique images: 5088


## Step 2: Split once (70 / 15 / 15, random_state = 42)
- 'stratify' = label + gender + age_group

In [3]:
df["group"] = df["label"] + "_" + df["gender"] + "_" + df["age_group"]

# oru group-la romba kammi images irundha split error tharum, appo label vachu mattum split panrom
if df["group"].value_counts().min() < 10:
    print("Sila groups romba chinna, label mattum vachu split panren")
    df["group"] = df["label"]

# 70% train, 30% rest
train_df, rest_df = train_test_split(df, test_size=0.30, stratify=df["group"], random_state=42)

# rest -> half val (15%), half test (15%)
val_df, test_df = train_test_split(rest_df, test_size=0.50, stratify=rest_df["group"], random_state=42)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

all_df = pd.concat([train_df, val_df, test_df])
all_df.to_csv(DATA_DIR / "split_info.csv", index=False)    # which image is in which split

print("train:", len(train_df), " val:", len(val_df), " test:", len(test_df))

train: 3561  val: 763  test: 764


## Step 3: Create the three split folders

In [4]:
def make_folders(folder_name, column):
    root = DATA_DIR / folder_name
    shutil.rmtree(root, ignore_errors=True)              # old folder irundha delete (thirumba run pannalaam)

    missing = 0
    for i in range(len(all_df)):
        image_id = all_df.iloc[i]["image_id"]
        split = all_df.iloc[i]["split"]                  # train / val / test
        cls = all_df.iloc[i][column]                     # real / male / 18-25 ...

        source = UNIQUE_DIR / f"{image_id}.jpg"
        if not source.exists():
            missing += 1
            continue

        target_folder = root / split / cls
        target_folder.mkdir(parents=True, exist_ok=True)
        shutil.copy(source, target_folder / f"{image_id}.jpg")

    print(folder_name, "done | missing images:", missing)

make_folders("split_real_fake", "label")
make_folders("split_gender", "gender")
make_folders("split_age", "age_group")

split_real_fake done | missing images: 0
split_gender done | missing images: 0
split_age done | missing images: 0


## Step 4: Check the counts

In [9]:
print(pd.crosstab(all_df["label"], all_df["split"]), "\n")
print("-" * 50)
print(pd.crosstab(all_df["gender"], all_df["split"]), "\n")
print("-" * 50)
print(pd.crosstab(all_df["age_group"], all_df["split"]))

split  test  train  val
label                  
fake    410   1923  413
real    354   1638  350 

--------------------------------------------------
split    test  train  val
gender                   
female    131    616  133
male      130    605  130
unknown   503   2340  500 

--------------------------------------------------
split      test  train  val
age_group                  
18-25       196    915  196
26-35       188    877  188
36-50       194    908  195
50+         186    861  184


In [6]:
# edhaavadhu image rendu splits-la irukka? (ellaam 0 varanum)
print("Overlap train/val :", len(set(train_df.image_id) & set(val_df.image_id)))
print("Overlap train/test:", len(set(train_df.image_id) & set(test_df.image_id)))
print("Overlap val/test  :", len(set(val_df.image_id) & set(test_df.image_id)))

Overlap train/val : 0
Overlap train/test: 0
Overlap val/test  : 0


## Step 5: Target
Input (X) = image. Target (y) = features

In [1]:
from pathlib import Path
import json

DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")
SPLITS = ["train", "val", "test"]

# Classes are in alphabetical order, because ImageFolder gives index numbers in this order.
targets = {
    "real_fake": {"folder": "split_real_fake", "classes": ["fake", "real"]},
    "gender":    {"folder": "split_gender",    "classes": ["female", "male", "unknown"]},
    "age":       {"folder": "split_age",       "classes": ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"]},
}

mapping = {}
for task, info in targets.items():
    classes = info["classes"]
    assert classes == sorted(classes), f"{task}: classes must be in alphabetical order"

    # check that every split has exactly these class folders
    for split in SPLITS:
        split_dir = DATA_DIR / info["folder"] / split
        found = sorted(f.name for f in split_dir.iterdir() if f.is_dir())
        if found != classes:
            print(f"WARNING {task}/{split}: found {found}, expected {classes}")

    mapping[task] = {name: i for i, name in enumerate(classes)}
    print(f"{task} -> folder: {info['folder']}")
    for name, i in mapping[task].items():
        print(f"     {name} = {i}")

# optional: save it so the Streamlit app can use the same mapping later
with open(DATA_DIR / "target_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

real_fake -> folder: split_real_fake
     fake = 0
     real = 1
gender -> folder: split_gender
     female = 0
     male = 1
     unknown = 2
age -> folder: split_age
     18-25 = 0
     26-35 = 1
     36-50 = 2
     50+ = 3
     not_applicable = 4
     under_18 = 5
